# Normaliser les mentions TIME sans coréférences

Ce notebook transforme les entités automatiques `TIME` en informations temporelles exploitables pour les triptyques.

Il ne s'appuie pas sur les `COREF_name`. L'objectif est volontairement plus difficile : partir de la mention brute repérée par Propp-fr et reconstruire, quand c'est possible, une année, un mois, un jour, une heure, ou un intervalle.

La logique générale est séquentielle. On lit les mentions `TIME` dans l'ordre du texte et on garde une ancre temporelle courante. Cette ancre permet de résoudre les expressions relatives comme `le lendemain`, `depuis deux jours`, `trois heures après` ou `à minuit précis`.

Le notebook ne force pas toutes les mentions à devenir des dates. Certaines expressions désignent une durée, une fréquence ou un moment vague. Dans ces cas, on garde l'information sous une forme plus prudente, par exemple `time_kind = duree` ou `time_kind = frequence`.

La sortie principale est ensuite utilisée par le notebook suivant, qui rattache les temps normalisés aux triptyques.


## 1) Paramètres et listes de référence

Cette section définit les chemins et les petits dictionnaires nécessaires aux règles de normalisation.

`DATE_DEBUT` sert à calculer le jour romanesque : si une mention est normalisée en `1872-10-02`, elle correspond au premier jour du roman. `ANNEE_DEFAUT` sert quand une mention donne un jour et un mois, mais pas l'année.

Les dictionnaires de mois, jours de semaine et nombres écrits en lettres permettent de traiter des formes comme `deux octobre`, `samedi 21 décembre`, `onze heures et demie` ou `quatre-vingts jours`.


In [13]:
import pandas as pd
import numpy as np
import re
import unicodedata
from datetime import datetime, timedelta

ENTITIES = '../data/PROPP/all_txt/tdm_auto_allchap.entities'
OUT = '../results/csv_triptyques/time_mentions_auto.csv'
OUT_EXEMPLES = '../results/csv_triptyques/time_mentions_exemples.csv'

# Point de départ utilisé pour calculer le jour romanesque.
# Le jour 1 du récit commence ici ; cela permet de créer une chronologie relative.
DATE_DEBUT = pd.Timestamp('1872-10-02')
ANNEE_DEFAUT = 1872

MONTHS = {
    'janvier': 1, 'fevrier': 2, 'mars': 3, 'avril': 4, 'mai': 5, 'juin': 6,
    'juillet': 7, 'aout': 8, 'septembre': 9, 'octobre': 10, 'novembre': 11, 'decembre': 12}

WEEKDAYS = {'lundi': 0, 'mardi': 1, 'mercredi': 2, 'jeudi': 3, 'vendredi': 4, 'samedi': 5, 'dimanche': 6}

# Approximation volontaire pour les moments vagues.
MOMENT_HOURS = {
    'aube': 6, 'lever du jour': 6, 'lever du soleil': 6, 'premieres heures': 5,
    'matin': 8, 'matinee': 9, 'journee': 12,
    'midi': 12, 'apres midi': 15, 'dejeuner': 12, 'diner': 19, 'souper': 20, 'repas': 12,
    'soir': 20, 'soiree': 20, 'nuit': 22, 'coucher': 22}

# Petits nombres flous : utiles pour ne pas perdre "quelques heures".
FUZZY_NUM = {'quelque': 3, 'quelques': 3, 'plusieurs': 4, 'certaines': 3, 'certain': 3, 'peu': 2}

# Faux positifs fréquents de l'annotation TIME automatique.
# Ces formes sont parfois annotées TIME alors qu'elles ne sont pas temporelles.
GARBAGE_PATTERNS = [
    r"^qui$", r"^laquelle$", r"^lequel$", r"^lesquels$", r"^on etait encore$",
    r"traite de", r"milles?", r"prison", r"conference", r"sacrifice", r"supplice",]

# Nombres écrits en lettres, nécessaires pour les dates, heures et durées.
# Nombres écrits en lettres.
# Les fonctions plus bas combinent ces valeurs pour lire 'vingt et un', 'quatre vingts', etc.
NUM_WORDS = {
    'zero': 0, 'un': 1, 'une': 1, 'deux': 2, 'trois': 3, 'quatre': 4, 'cinq': 5,
    'six': 6, 'sept': 7, 'huit': 8, 'neuf': 9, 'dix': 10, 'onze': 11, 'douze': 12,
    'treize': 13, 'quatorze': 14, 'quinze': 15, 'seize': 16, 'vingt': 20, 'trente': 30,
    'quarante': 40, 'cinquante': 50, 'soixante': 60}

# Mots ignorés quand on extrait un nombre depuis une expression.
STOP_NUM = {
    'et', 'de', 'd', 'du', 'des', 'le', 'la', 'les', 'l', 'a', 'au', 'aux', 'en', 'vers',
    'avant', 'apres', 'moins', 'plus', 'tard', 'tot', 'matin', 'soir', 'midi', 'minuit',
    'minute', 'minutes', 'heure', 'heures', 'jour', 'jours', 'an', 'ans', 'annee', 'annees',
    'mois', 'demi', 'demie', 'quart', 'quarts', 'environ', 'precis', 'precises', 'sonnant',
    'sonnaient', 'sonnait', 'seulement', 'auparavant', 'dernier', 'derniere', 'prochain', 'prochaine',}


## 2) Nettoyer le texte et lire les nombres

Avant de reconnaître les dates, le texte est simplifié : accents retirés, tirets normalisés, ponctuation supprimée, espaces nettoyés.

Cette étape est importante car les mentions Propp peuvent contenir plusieurs graphies équivalentes : `quatre-vingts`, `quatre vingts`, `l’année`, `l'annee`, etc.

Les fonctions de cette section servent aussi à convertir des nombres écrits en lettres en nombres ordinaires. C'est indispensable pour les heures, les durées et certains jours du mois.


In [14]:
# On met toutes les mentions au même format avant de chercher des motifs.
def norm(s):
    # On part de la mention originale, souvent bruitée par les accents ou la ponctuation.
    # La sortie sert seulement aux règles de détection, pas à l'export final.
    s = '' if pd.isna(s) else str(s).lower()
    # Les accents sont retirés pour comparer 'février' et 'fevrier' de la même manière.
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(c for c in s if not unicodedata.combining(c))
    s = s.replace('’', "'").replace('`', "'")
    s = s.replace('quatre-vingts', 'quatre vingts').replace('quatre-vingt', 'quatre vingt')
    s = s.replace('dix-sept', 'dix sept').replace('dix-huit', 'dix huit').replace('dix-neuf', 'dix neuf')
    s = re.sub(r"[.,;:!?()\[\]«»]", ' ', s)
    s = s.replace('-', ' ')
    # On termine par un espace unique pour faciliter les regex.
    s = re.sub(r"\s+", ' ', s).strip()
    return s


# On garde seulement les mots qui peuvent former un nombre.
def clean_num_text(s):
    # On extrait la partie numérique d'une expression.
    # Exemple : 'deux jours après' devient surtout 'deux'.
    s = norm(s)
    s = re.sub(r"\bou\b.*$", '', s).strip()
    toks = [t for t in s.split() if t not in STOP_NUM]
    return toks


# Convertit 'vingt et un', 'quatre-vingts' ou '12' en nombre.
def parse_num_phrase(s):
    # Convertit une petite expression numérique française en entier.
    # La fonction couvre les besoins du roman : jours, heures, durées et années simples.
    # utiles aux dates, heures et durées, sans chercher à être un analyseur complet du français.
    if s is None:
        return None
    s0 = norm(s)
    if re.fullmatch(r"\d+(?:/\d+)?", s0):
        if '/' in s0:
            a, b = s0.split('/')
            return float(a) / float(b)
        return int(s0)
    if s0 in FUZZY_NUM:
        return FUZZY_NUM[s0]
    toks = clean_num_text(s0)
    # Si aucun mot numérique ne reste, on ne peut rien convertir.
    if not toks:
        return None
    if len(toks) == 1 and toks[0] in FUZZY_NUM:
        return FUZZY_NUM[toks[0]]

    total = 0
    current = 0
    seen = False
    i = 0
    # On parcourt les mots de gauche à droite pour additionner les blocs numériques.
    while i < len(toks):
        t = toks[i]
        if re.fullmatch(r"\d+", t):
            current += int(t); seen = True; i += 1; continue
        if t == 'dix' and i + 1 < len(toks) and toks[i + 1] in {'sept', 'huit', 'neuf'}:
            current += 10 + NUM_WORDS[toks[i + 1]]; seen = True; i += 2; continue
        if t == 'soixante' and i + 1 < len(toks):
            nxt = toks[i + 1]
            if nxt == 'dix':
                add = 70
                if i + 2 < len(toks) and toks[i + 2] in {'un', 'une', 'deux', 'trois', 'quatre', 'cinq', 'six', 'sept', 'huit', 'neuf'}:
                    add += NUM_WORDS[toks[i + 2]]; i += 3
                else:
                    i += 2
                current += add; seen = True; continue
            if nxt in {'onze', 'douze', 'treize', 'quatorze', 'quinze', 'seize'}:
                current += 60 + NUM_WORDS[nxt]; seen = True; i += 2; continue
        if t == 'quatre' and i + 1 < len(toks) and toks[i + 1].startswith('vingt'):
            add = 80; i += 2
            if i < len(toks):
                if toks[i] == 'dix':
                    add += 10; i += 1
                    if i < len(toks) and toks[i] in {'un', 'une', 'deux', 'trois', 'quatre', 'cinq', 'six', 'sept', 'huit', 'neuf'}:
                        add += NUM_WORDS[toks[i]]; i += 1
                elif toks[i] in NUM_WORDS and NUM_WORDS[toks[i]] < 10:
                    add += NUM_WORDS[toks[i]]; i += 1
            current += add; seen = True; continue
        if t == 'cent':
            current = (current if current else 1) * 100; seen = True; i += 1; continue
        if t == 'mille':
            total += (current if current else 1) * 1000; current = 0; seen = True; i += 1; continue
        if t in NUM_WORDS:
            current += NUM_WORDS[t]; seen = True; i += 1; continue
        i += 1
    return total + current if seen else None



# Cherche un nombre juste avant un mot comme 'heures'.
def suffix_num_before(tokens, idx, max_len=6):
    for l in range(max_len, 0, -1):
        start = idx - l
        if start < 0:
            continue
        val = parse_num_phrase(' '.join(tokens[start:idx]))
        if val is not None:
            return val
    return None


# Cherche un nombre juste après un mot comme 'midi' ou 'heures'.
def first_num_after(tokens, idx, max_len=6):
    seg = []
    for t in tokens[idx + 1:idx + 1 + max_len]:
        if t in {'du', 'de', 'des', 'd', 'le', 'la', 'les', 'matin', 'soir', 'minutes', 'minute', 'precis', 'precises', 'sonnant', 'sonnaient', 'sonnait', 'apres', 'midi', 'avant'} and seg:
            break
        seg.append(t)
    for l in range(len(seg), 0, -1):
        val = parse_num_phrase(' '.join(seg[:l]))
        if val is not None:
            return val
    return None


## 3) Lire les heures, dates et temps relatifs

Cette section contient les règles qui reconnaissent les dates et les heures les plus fréquentes.

On distingue plusieurs cas : une date absolue (`2 octobre`, `en 1814`), une heure (`à minuit précis`), un intervalle horaire (`entre onze heures et onze heures et demie`) et un relatif (`le lendemain`, `trois jours après`).

Les expressions relatives ne peuvent être calculées que si une ancre existe déjà. Par exemple, `le lendemain` n'a pas de valeur en soi, mais devient calculable si le contexte courant est déjà `1872-10-02`.


In [15]:
# Lit 'moins le quart' ou 'moins vingt'.
def parse_minus_value(words):
    # Lit la partie située après 'moins'.
    # Exemple : 'midi moins vingt' doit donner 20 minutes à retrancher.
    phrase = ' '.join(words)
    if 'quart' in phrase:
        return 15
    if 'demi' in phrase or 'demie' in phrase:
        return 30
    return parse_num_phrase(phrase)


# Reconnaît les heures : minuit, midi, 11h30, onze heures et demie, etc.
def parse_clock(n):
    # Reconnaît une heure dans une mention normalisée.
    # La fonction renvoie seulement (heure, minute), sans choisir la date.
    # Renvoie (heure, minute) quand la mention contient une heure lisible.
    # Ne décide pas encore à quelle date rattacher cette heure.
    tokens = n.split()
    # On travaille avec les mots pour gérer les formes écrites en toutes lettres.
    hour = None
    minute = 0
    # Cas spéciaux : minuit et midi se lisent mal comme des nombres ordinaires.
    # 'minuit' et 'midi' sont traités avant les nombres ordinaires.
    if 'minuit' in tokens:
        hour = 0
        idx = tokens.index('minuit')
        after = tokens[idx + 1:idx + 6]
        if after and after[0] == 'moins':
            val = parse_minus_value(after[1:])
            if val is not None:
                hour = 23
                minute = 60 - int(val)
        else:
            val = first_num_after(tokens, idx)
            if val is not None and val < 60:
                minute = int(val)
    elif 'midi' in tokens:
        hour = 12
        idx = tokens.index('midi')
        after = tokens[idx + 1:idx + 6]
        if after and after[0] == 'moins':
            val = parse_minus_value(after[1:])
            if val is not None:
                hour = 11
                minute = 60 - int(val)
        else:
            val = first_num_after(tokens, idx)
            if val is not None and val < 60:
                minute = int(val)
    else:
        m = re.search(r"\b(\d{1,2})\s*h\s*(\d{1,2})?\b", n)
        if m:
            hour = int(m.group(1))
            minute = int(m.group(2) or 0)
        else:
            idxs = [i for i, t in enumerate(tokens) if t in ('heure', 'heures')]
            if not idxs:
                return None
            idx = idxs[0]
            h = suffix_num_before(tokens, idx)
            if h is None or h >= 24:
                return None
            hour = int(h)
            after = tokens[idx + 1:idx + 7]
            if after[:2] in [['et', 'demie'], ['et', 'demi']]:
                minute = 30
            elif after[:2] == ['et', 'quart'] or after[:3] == ['et', 'un', 'quart']:
                minute = 15
            elif after and after[0] == 'moins':
                val = parse_minus_value(after[1:5])
                if val is not None:
                    hour = (hour - 1) % 24
                    minute = 60 - int(val)
            else:
                val = first_num_after(tokens, idx)
                if val is not None and val < 60:
                    minute = int(val)
    if hour is None:
        return None
    # On corrige les heures ambiguës selon matin / soir.
    if any(x in n for x in ['du soir', 'le soir', 'soir', 'apres midi', 'apres-midi']):
        if hour < 12:
            hour += 12
    if any(x in n for x in ['du matin', 'le matin']) and hour == 12:
        hour = 0
    return hour % 24, minute


# Test rapide : est-ce plutôt une durée qu'une heure ou une date ?
def looks_like_duration(n):
    # Détecte les mentions qui contiennent une unité de durée.
    # Cela évite de lire 'vingt-quatre heures' comme une simple heure.
    if re.search(r"\b(milles?|kilometres?)\b", n):
        return False
    if any(x in n for x in ['pendant', 'durant', 'sur ', 'de retard', 'd avance', 'd arret', 'de marche', 'de repos', 'de prison']):
        return True
    if parse_clock(n) is not None:
        return False
    if re.search(r"\b(en|durant|pendant)\b.*\b(jours?|heures?|minutes?|secondes?|ans?|mois|siecle|siecles)\b", n):
        if re.search(r"\b(1[5-9]\d{2}|20\d{2})\b", n) and 'jour' not in n and 'heure' not in n and 'minute' not in n and 'ans' not in n:
            return False
        return True
    num_words = r"un|une|deux|trois|quatre|cinq|six|sept|huit|neuf|dix|onze|douze|treize|quatorze|quinze|seize|vingt|trente|quarante|cinquante|soixante|quatre vingts|quelques|plusieurs|certaines|des|les|derniers|dernieres|premiers|premieres|\d+|\d+e"
    if re.search(r"\b(?:" + num_words + r")\b.*\b(jours?|heures?|minutes?|secondes?|ans?|annees?|mois|siecle|siecles)\b", n):
        return True
    if re.search(r"\bde [a-z0-9 ]+ a [a-z0-9 ]+ jours?\b", n):
        return True
    return False

# Reconnaît des intervalles horaires simples : entre 11h et 11h30.
def parse_clock_interval(n):
    # Reconnaît deux heures dans une même mention.
    # Exemple : 'entre onze heures et onze heures et demie'.
    if n.startswith('entre ') and ' et ' in n:
        p1, p2 = n.replace('entre ', '', 1).split(' et ', 1)
    elif n.startswith('de ') and ' a ' in n:
        p1, p2 = n.replace('de ', '', 1).split(' a ', 1)
    else:
        return None
    c1 = parse_clock(p1)
    c2 = parse_clock(p2)
    if c1 and c2:
        return c1, c2
    return None


# Code compact avec 00 pour les éléments inconnus.
def make_code(y=None, mo=None, d=None, h=None, mi=None):
    if y is None or pd.isna(y):
        return np.nan
    return f"{int(y):04d}-{int(mo or 0):02d}-{int(d or 0):02d}-{int(h or 0):02d}-{int(mi or 0):02d}"

def day_rom(dt):
    # Calcule le rang du jour dans le récit à partir de DATE_DEBUT.
    if dt is None:
        return np.nan
    return (pd.Timestamp(dt.date()) - DATE_DEBUT).days + 1

def fmt_dt(dt, precision):
    # Formate les dates pour le CSV selon leur précision.
    if dt is None:
        return np.nan
    if precision in ('date', 'jour'):
        return f"{dt.year:04d}-{dt.month:02d}-{dt.day:02d}"
    if precision == 'datetime':
        return f"{dt.year:04d}-{dt.month:02d}-{dt.day:02d} {dt.hour:02d}:{dt.minute:02d}"
    return str(dt.year)

def should_update_context(year, narrative_year):
    # Décide si une date peut devenir la nouvelle ancre narrative.
    # Une année historique isolée ne doit pas forcément déplacer tout le récit.
    if year is None:
        return False
    if narrative_year is None:
        return True
    return int(year) >= int(narrative_year) - 1


# Intervalles avec deux dates explicites.
def absolute_interval(n, cur_year, cur_month):
    y = cur_year or ANNEE_DEFAUT
    m = re.search(r"depuis le (\d{1,2}) ([a-z]+) jusqu au (\d{1,2}) ([a-z]+)", n)
    if m and m.group(2) in MONTHS and m.group(4) in MONTHS:
        return datetime(y, MONTHS[m.group(2)], int(m.group(1))), datetime(y, MONTHS[m.group(4)], int(m.group(3))), 'depuis_jusqu_date'
    m = re.search(r"(?:nuit|journee|journees)? ?du (\d{1,2}) au (\d{1,2}) ([a-z]+)", n)
    if m and m.group(3) in MONTHS:
        return datetime(y, MONTHS[m.group(3)], int(m.group(1))), datetime(y, MONTHS[m.group(3)], int(m.group(2))), 'du_au_date'
    m = re.search(r"du (\d{1,2}).*au (\d{1,2})", n)
    if m and cur_month:
        return datetime(y, cur_month, int(m.group(1))), datetime(y, cur_month, int(m.group(2))), 'du_au_jour'
    return None


# Dates absolues : année seule, date complète, mois approximatif ou jour seul.
def absolute_date(n, cur_year, cur_month):
    # On lit d'abord les dates complètes, puis les dates plus partielles.
    for month, mo in MONTHS.items():
        m = re.search(r"(?:le|ce|du|au|date du|journee du|lundi|mardi|mercredi|jeudi|vendredi|samedi|dimanche)?\s*(\d{1,2})(?:er)?\s+" + month + r"(?:\s+(\d{4}))?", n)
        if m:
            y = int(m.group(2)) if m.group(2) else (cur_year or ANNEE_DEFAUT)
            d = int(m.group(1))
            try:
                return datetime(y, mo, d), 'date_complete'
            except ValueError:
                return None
    for month, mo in MONTHS.items():
        if month in n and re.search(r"\b(premier|premiers|debut|fin|milieu)\b", n):
            y = cur_year or ANNEE_DEFAUT
            return {'year': y, 'month': mo, 'precision': 'mois'}, 'mois_approx'
    m = re.search(r"\b(1[5-9]\d{2}|20\d{2})\b", n)
    if m:
        return {'year': int(m.group(1)), 'precision': 'annee'}, 'annee_seule'
    m = re.fullmatch(r"(?:le|au|du|ce)?\s*(\d{1,2})", n)
    if m and cur_year and cur_month:
        try:
            return datetime(cur_year, cur_month, int(m.group(1))), 'jour_seul'
        except ValueError:
            return None
    return None

def parse_anchor_dt(cur_dt, cur_date):
    # Choisit la meilleure ancre disponible pour calculer un relatif.
    # On préfère une date-heure complète, puis une date sans heure.
    if cur_dt is not None:
        return cur_dt
    if cur_date is not None:
        return datetime(cur_date.year, cur_date.month, cur_date.day)
    return None

def apply_moment_vague(n, base):
    if base is None:
        return None
    hour = None
    rule = None
    for key, h in MOMENT_HOURS.items():
        if key in n:
            hour = h
            rule = 'moment_' + key.replace(' ', '_')
            break
    if hour is None:
        return None
    if 'lendemain' in n or 'suivante' in n or 'suivant' in n:
        base = base + timedelta(days=1)
    if 'veille' in n or 'precedente' in n or 'precedent' in n:
        base = base - timedelta(days=1)
    return base.replace(hour=hour, minute=0), rule

def parse_weekday_expr(n, base):
    # Résout les jours de semaine relatifs au contexte.
    # Exemple : 'mercredi dernier' dépend de la date courante.
    if base is None:
        return None
    found = [w for w in WEEKDAYS if re.search(r"\b" + w + r"\b", n)]
    if not found:
        return None
    w = found[0]
    target = WEEKDAYS[w]
    delta = (target - base.weekday()) % 7
    if 'dernier' in n or 'derniere' in n:
        delta = -((base.weekday() - target) % 7 or 7)
    elif 'prochain' in n or 'prochaine' in n:
        delta = ((target - base.weekday()) % 7 or 7)
    dt = base + timedelta(days=delta)
    mv = apply_moment_vague(n, dt)
    if mv:
        dt, _ = mv
    return dt, 'jour_semaine_' + ('dernier' if 'dernier' in n else 'prochain' if 'prochain' in n else 'proche')


# Expressions relatives résolues à partir de l'ancre courante.
def relative(n, cur_dt, cur_date, cur_year, cur_month, lookahead_dt=None):
    # Résout les expressions relatives calculables.
    # La fonction peut utiliser le contexte précédent ou une date explicite plus loin dans la même phrase.
    # Les relatifs ne sont calculables que si une ancre temporelle existe déjà.
    base = parse_anchor_dt(cur_dt, cur_date)

    # Références au moment courant : aujourd'hui, ce jour-là, en ce moment.
    if re.search(r"\bce jour(?:\s+la)?(?:\s+meme)?\b", n) or 'aujourd hui' in n or "aujourd'hui" in n or 'cette journee' in n:
        if base is not None:
            return base, 'deictique_date_courante'
        if lookahead_dt is not None:
            return lookahead_dt, 'deictique_date_suivante_meme_phrase'
        return None

    if base is None:
        return None
    if any(k in n for k in ['en ce moment', 'a ce moment', 'a cet instant', 'a partir de ce moment', 'au moment', 'cette heure', "l' heure", 'l heure']):
        return base, 'deictique_instant_courant'
    # Depuis/jusqu'à + heure : on traite l'heure comme un repère horaire, pas comme une durée.
    if n.startswith('depuis ') or n.startswith('jusqu'):
        cl = parse_clock(n)
        if cl and cur_date is not None:
            h, mi = cl
            return datetime(cur_date.year, cur_date.month, cur_date.day, h, mi), 'heure_depuis_jusqu'

    if "l' annee" in n or 'l annee' in n or 'cette annee' in n:
        y = cur_year or base.year
        return {'year': y, 'precision': 'annee'}, 'annee_courante'
    if n in {'le mois', 'ce mois'} and cur_year and cur_month:
        return {'year': cur_year, 'month': cur_month, 'precision': 'mois'}, 'mois_courant'
    if n in {'le jour', 'le quantieme', 'au jour fixe'}:
        return base, 'jour_courant'

    # Jours de la semaine : mercredi, mercredi dernier, samedi soir.
    wd = parse_weekday_expr(n, base)
    if wd:
        return wd

    # Relatifs en jours : demain, la veille, le lendemain.
    if 'surlendemain' in n:
        return base + timedelta(days=2), 'lendemain_plus_2'
    if 'lendemain' in n or 'demain' in n or 'au lendemain' in n:
        return base + timedelta(days=1), 'lendemain_plus_1'
    if 'avant hier' in n:
        return base - timedelta(days=2), 'veille_moins_2'
    if 'veille' in n or 'hier' in n:
        return base - timedelta(days=1), 'veille_moins_1'

    # Relatifs flous. On garde une approximation simple, toujours vérifiable ensuite.
    fuzzy_offsets = [
        (r"quelques instants (apres|plus tard)", 1, '1min'),
        (r"un instant (apres|plus tard)", 1, '1min'),
        (r"quelques minutes (apres|plus tard)", 5, '5min'),
        (r"quelques heures (apres|plus tard)", 180, '3h'),
        (r"quelques jours (apres|plus tard)", 4320, '3j'),
        (r"dans quelques heures", 180, '3h'),
        (r"dans quelques minutes", 5, '5min'),
    ]
    for pat, minutes, label in fuzzy_offsets:
        if re.search(pat, n):
            return base + timedelta(minutes=minutes), f'offset_flou_{label}'

    # Décalages exacts en jours, y compris 'il y a' et 'voilà'.
    m = re.search(r"(?:dans|apres) ([a-z0-9 ]{1,40}) jours?", n)
    if m:
        val = parse_num_phrase(m.group(1))
        if val is not None:
            return base + timedelta(days=int(val)), f'offset_{int(val)}j'
    m = re.search(r"(?:avant|il y a|voila) ([a-z0-9 ]{1,40}) jours?", n)
    if m:
        val = parse_num_phrase(m.group(1))
        if val is not None:
            return base - timedelta(days=int(val)), f'offset_{-int(val)}j'
    m = re.search(r"([a-z0-9 ]{1,40}) jours? (apres|plus tard|auparavant|avant)", n)
    if m:
        val = parse_num_phrase(m.group(1))
        if val is not None:
            sign = -1 if m.group(2) in ('auparavant', 'avant') else 1
            return base + timedelta(days=sign * int(val)), f'offset_{sign * int(val)}j'

    # Décalages exacts en minutes ou en heures.
    patterns = [
        r"(?:dans|apres|avant|depuis|il y a|voila) ([a-z0-9 ]{1,40}) (minutes?|heures?)",
        r"([a-z0-9 ]{1,40}) (minutes?|heures?) (apres|plus tard|plus tot|auparavant|avant)",
        r"(une demi heure|un quart d heure|trois quarts d heure) (apres|plus tard|plus tot|auparavant|avant)",
        r"dans (une demi heure|un quart d heure|trois quarts d heure)",
    ]
    for pat in patterns:
        m = re.search(pat, n)
        if not m:
            continue
        groups = m.groups()
        phrase = groups[0]
        unit = groups[1] if len(groups) > 1 else ''
        direction = ' '.join(groups)
        if 'demi' in phrase:
            minutes = 30
        elif 'trois quarts' in phrase:
            minutes = 45
        elif 'quart' in phrase:
            minutes = 15
        else:
            val = parse_num_phrase(phrase)
            if val is None:
                continue
            minutes = val if unit.startswith('minute') else val * 60
        if 'depuis' in n:
            return base, f'depuis_duree_{int(minutes)}min'
        sign = -1 if any(x in direction for x in ['avant', 'plus tot', 'auparavant', 'il y a', 'voila']) else 1
        return base + timedelta(minutes=sign * int(minutes)), f'offset_{sign * int(minutes)}min'

    # Recul en années : il y a cinq ans, voilà cinq ans.
    m = re.search(r"(?:depuis|il y a|voila|en) ([a-z0-9 ]{1,40}) ans?", n)
    if m and cur_year:
        val = parse_num_phrase(m.group(1))
        if val is not None:
            return {'year': cur_year - int(val), 'precision': 'annee'}, f'annee_moins_{int(val)}'

    # Moments vagues de la journée, seulement s'il n'y a pas déjà une vraie heure.
    if not looks_like_duration(n) and parse_clock(n) is None and 'chaque' not in n and 'tous les' not in n:
        mv = apply_moment_vague(n, base)
        if mv:
            return mv

    return None


## 4) Lire les durées

Les durées sont traitées séparément parce qu'elles ne désignent pas toujours un instant.

Une durée simple comme `7 jours` ou `quatre-vingts jours` donne surtout une quantité. Elle ne doit pas devenir automatiquement une date.

Une durée ancrée comme `depuis deux jours`, `pendant vingt-quatre heures` ou `voilà cinq ans` peut produire un intervalle. Dans ce cas, le notebook calcule un début et une fin avec `debut_auto` et `fin_auto`.

Cette distinction est importante pour la visualisation : une date ponctuelle et un intervalle ne doivent pas être utilisés exactement de la même façon.


In [16]:
# Unités de durée reconnues dans les mentions TIME.
DURATION_UNIT_RE = r"secondes?|instants?|minutes?|heures?|jours?|semaines?|mois|ans?|annees?|siecles?"


# On ramène les pluriels et variantes vers une unité canonique.
UNIT_CANON = {
    'seconde': 'seconde', 'secondes': 'seconde',
    'instant': 'minute', 'instants': 'minute',
    'minute': 'minute', 'minutes': 'minute',
    'heure': 'heure', 'heures': 'heure',
    'jour': 'jour', 'jours': 'jour',
    'semaine': 'semaine', 'semaines': 'semaine',
    'mois': 'mois',
    'an': 'annee', 'ans': 'annee', 'annee': 'annee', 'annees': 'annee',
    'siecle': 'siecle', 'siecles': 'siecle',}


# Décale une date selon une durée, sans transformer les mois/années en minutes.
def shift_datetime(dt, value, unit, sign=1):
    # Décale une date selon une durée.
    # Les jours/heures utilisent timedelta ; les mois/années sont ajustés à part.
    # On décale l'ancre courante sans convertir les mois ou les années en jours fixes.
    if dt is None or value is None or pd.isna(value):
        return None
    value = float(value) * sign
    if unit == 'seconde':
        return dt + timedelta(seconds=value)
    if unit == 'minute':
        return dt + timedelta(minutes=value)
    if unit == 'heure':
        return dt + timedelta(hours=value)
    if unit == 'jour':
        return dt + timedelta(days=value)
    if unit == 'semaine':
        return dt + timedelta(weeks=value)
    if unit == 'mois':
        return (pd.Timestamp(dt) + pd.DateOffset(months=int(value))).to_pydatetime()
    if unit == 'annee':
        return (pd.Timestamp(dt) + pd.DateOffset(years=int(value))).to_pydatetime()
    if unit == 'siecle':
        return (pd.Timestamp(dt) + pd.DateOffset(years=int(value) * 100)).to_pydatetime()
    return None

def parse_duration_info(n):
    # Extrait la quantité et l'unité d'une durée.
    # Exemple : 'depuis deux jours' -> 2 + jour + relation depuis.
    # Extrait la quantité et l'unité d'une durée, sans décider encore si elle est ancrée.
    if re.search(r"\b(milles?|kilometres?)\b", n):
        return None
    if parse_clock(n) is not None and not re.search(r"\b(retard|avance|arret|marche|repos|prison)\b", n):
        return None

    # Formes fixes : une demi-heure, un quart d'heure, trois quarts d'heure.
    if re.search(r"\bune demi heure\b", n):
        return {'duration_value': 0.5, 'duration_unit': 'heure', 'duration_relation': duration_relation(n)}
    if re.search(r"\bun quart d heure\b", n):
        return {'duration_value': 0.25, 'duration_unit': 'heure', 'duration_relation': duration_relation(n)}
    if re.search(r"\btrois quarts d heure\b", n):
        return {'duration_value': 0.75, 'duration_unit': 'heure', 'duration_relation': duration_relation(n)}

    # Intervalles de quantité : de 7 à 8 jours.
    # On garde la borne haute pour calculer un intervalle si une ancre existe.
    # On cherche une quantité suivie d'une unité temporelle.
    m = re.search(r"\bde ([a-z0-9 ]{1,40}) a ([a-z0-9 ]{1,40}) (" + DURATION_UNIT_RE + r")\b", n)
    if m:
        v1 = parse_num_phrase(m.group(1))
        v2 = parse_num_phrase(m.group(2))
        unit = UNIT_CANON.get(m.group(3), m.group(3))
        if v2 is not None:
            return {
                'duration_value': float(v2), 'duration_value_min': np.nan if v1 is None else float(v1),
                'duration_unit': unit,
                'duration_relation': duration_relation(n),
            }

    # Cas flous sans nombre exact.
    if re.search(r"\b(de longues annees|longues annees|nombreuses annees)\b", n):
        return {'duration_value': np.nan, 'duration_unit': 'annee', 'duration_relation': duration_relation(n)}

    m = re.search(r"\b([a-z0-9 ]{1,60}) (" + DURATION_UNIT_RE + r")\b", n)
    if not m:
        return None
    phrase = m.group(1)
    unit = UNIT_CANON.get(m.group(2), m.group(2))
    val = parse_num_phrase(phrase)
    if val is None:
        return None
    return {
        'duration_value': float(val),
        'duration_unit': unit,
        'duration_relation': duration_relation(n),
    }



# Ces formes indiquent plutôt un point relatif qu'une durée-intervalle.
def is_relative_offset_duration(n):
    # Distingue 'trois jours après' d'une durée comme 'pendant trois jours'.
    # Le premier cas désigne une date relative ponctuelle.
    return bool(re.search(r"\b(dans|apres|plus tard|plus tot|auparavant|avant|il y a)\b", n))


# Relation temporelle portée par la durée : depuis, pendant, en, etc.
def duration_relation(n):
    # Identifie la relation temporelle portée par la durée.
    # Cette relation décide si l'intervalle va vers le passé ou vers l'avenir.
    if 'depuis' in n or re.search(r"\bvoila\b", n):
        return 'depuis'
    if 'retard' in n:
        return 'retard'
    if 'avance' in n:
        return 'avance'
    if 'pendant' in n or 'durant' in n:
        return 'pendant'
    if n.startswith('sur ') or ' sur ' in n:
        return 'sur'
    if re.search(r"^en\b", n):
        return 'en'
    return 'duree_simple'


# Si une durée est ancrée, on calcule un début et une fin.
def anchored_duration_interval(n, base):
    # Transforme une durée en intervalle quand une ancre existe.
    # Si la durée n'est pas ancrée, on garde seulement la quantité.
    # Transforme certaines durées en intervalle quand une ancre temporelle existe.
    info = parse_duration_info(n)
    if info is None:
        return None
    if base is None or pd.isna(info.get('duration_value', np.nan)):
        return {'info': info, 'start': None, 'end': None, 'rule': 'duree_non_ancree'}

    val = info['duration_value']
    unit = info['duration_unit']
    relation = info['duration_relation']

    if relation in {'depuis'}:
        start = shift_datetime(base, val, unit, sign=-1)
        end = base
        rule = 'duree_depuis_intervalle'
    elif relation in {'pendant', 'durant', 'sur', 'en'}:
        start = base
        end = shift_datetime(base, val, unit, sign=1)
        rule = 'duree_forward_intervalle_' + relation
    else:
        start = None
        end = None
        rule = 'duree_quantite'

    return {'info': info, 'start': start, 'end': end, 'rule': rule}


## 5) Construire la table normalisée

La fonction principale applique les règles dans un ordre précis. Cet ordre évite plusieurs confusions.

Par exemple, `vingt-quatre heures` ressemble à une heure, mais c'est souvent une durée. De même, `en 1814` est une date absolue, tandis que `en quatre-vingts jours` est une durée.

Le notebook commence donc par les cas les plus structurés, puis passe aux cas plus généraux : intervalles, dates absolues, fréquences, durées, relatifs, heures seules, puis classement des cas non reconnus.



In [17]:
# Dernier filet de sécurité pour les mentions non reconnues.
def classify_leftover(n):
    # Arrive seulement après les règles principales.
    # L'objectif est de ne pas perdre l'information, même quand on ne peut pas dater.
    # Cette fonction classe ce qu'on ne peut pas normaliser proprement.
    if any(re.search(p, n) for p in GARBAGE_PATTERNS):
        return 'garbage_probable', 'non_temporel'
    if any(x in n for x in ['chaque jour', 'tous les jours', 'jour par jour', 'tous les mois', 'chaque mois']):
        return 'frequence', 'non_instancie'
    if re.search(r"\ba des heures\b|\ba de certaines heures\b", n):
        return 'frequence', 'non_instancie'
    if looks_like_duration(n):
        return 'duree', 'non_instancie'
    if any(x in n for x in ['depart', 'arrivee', 'evenement', 'revolte', 'meeting']):
        return 'reference_evenement', 'non_instancie'
    if re.search(r"\b(saison|hiver|printemps|ete|automne|epoque)\b", n):
        return 'moment_vague', 'saison'
    if any(x in n for x in ['matin', 'soir', 'nuit', 'journee', 'aube', 'crepuscule', 'lever du jour', 'lever du soleil', 'dejeuner', 'diner', 'souper', 'repas', 'moment', 'instant', 'heure', 'temps', 'apparition du jour', 'approche du jour', 'jour naissant', 'quartier de la lune']):
        return 'moment_vague', 'non_instancie'
    if any(x in n for x in ['depuis', 'jusqu', 'avant', 'apres', 'dans', 'plus tard', 'plus tot', 'ecoules']):
        return 'relatif_non_calcule', 'non_instancie'
    return 'non_parse', 'non_instancie'


# Fonction principale : elle lit les TIME dans l'ordre du texte.
def parse_time_table(time):
    # Elle parcourt les TIME dans l'ordre du texte et maintient le contexte courant.
    # Dates explicites qui apparaissent plus loin dans la même phrase.
    # Utile pour des cas comme : "ce jour-là même, 2 octobre".
    sentence_next_dates = {}
    # Ce dictionnaire aide à résoudre certains déictiques intra-phrastiques.
    for _, rr in time.iterrows():
        nn = norm(rr['text'])
        xx = absolute_date(nn, ANNEE_DEFAUT, None)
        if isinstance(xx, tuple) and isinstance(xx[0], datetime):
            sentence_next_dates.setdefault((int(rr.paragraph_ID), int(rr.sentence_ID)), xx[0])

    # Contexte courant utilisé pour résoudre les expressions relatives.
    # Ces variables représentent l'ancre temporelle courante.
    # Elles changent seulement quand une mention est jugée assez solide.
    narrative_year = None
    cur_year = None
    cur_month = None
    cur_date = None
    cur_dt = None
    rows = []

    # La boucle suivante avance dans l'ordre du texte et met à jour l'ancre courante.
    for i, r in time.iterrows():
        txt = r['text']
        n = norm(txt)
        anchor = fmt_dt(cur_dt, 'datetime') if cur_dt is not None else (
            fmt_dt(datetime(cur_date.year, cur_date.month, cur_date.day), 'date') if cur_date is not None else (
                str(cur_year) if cur_year else np.nan
            )
        )
        # Chaque mention TIME produit une ligne structurée.
        # Les champs non connus restent vides plutôt que d'être inventés.
        rec = {
            'entity_id': i,
            'start_token': int(r.start_token),
            'end_token': int(r.end_token),
            'paragraph_ID': int(r.paragraph_ID),
            'sentence_ID': int(r.sentence_ID),
            'text': txt,
            'text_norm': n,
            'time_kind': None,
            'precision': None,
            'annee': np.nan,
            'mois': np.nan,
            'jour': np.nan,
            'heure': np.nan,
            'minute': np.nan,
            'datetime_auto': np.nan,
            'time_code_auto': np.nan,
            'jour_romanesque': np.nan,
            'debut_auto': np.nan,
            'fin_auto': np.nan,
            'duration_value': np.nan,
            'duration_unit': np.nan,
            'duration_relation': np.nan,
            'anchor_before': anchor,
            'regle': ''}
        
        done = False

        # Chaque bloc ci-dessous essaie une famille de formes temporelles.
        # L'ordre compte : on traite les formes les plus spécifiques avant les plus générales.

        # 1. Intervalles horaires : entre onze heures et midi.
        ci = parse_clock_interval(n)
        if ci and cur_date is not None:
            (h1, mi1), (h2, mi2) = ci
            start_dt = datetime(cur_date.year, cur_date.month, cur_date.day, h1, mi1)
            end_dt = datetime(cur_date.year, cur_date.month, cur_date.day, h2, mi2)
            rec.update({
                'time_kind': 'intervalle', 'precision': 'heure',
                'annee': start_dt.year, 'mois': start_dt.month, 'jour': start_dt.day,
                'heure': h1, 'minute': mi1,
                'datetime_auto': fmt_dt(start_dt, 'datetime'),
                'time_code_auto': make_code(start_dt.year, start_dt.month, start_dt.day, h1, mi1),
                'jour_romanesque': day_rom(start_dt),
                'debut_auto': fmt_dt(start_dt, 'datetime'),
                'fin_auto': fmt_dt(end_dt, 'datetime'),
                'regle': 'intervalle_heure',
            })
            # L'intervalle commence à start_dt ; on l'utilise comme nouveau point courant.
            cur_dt = start_dt
            done = True

        # 2. Intervalles avec deux dates explicites.
        x = absolute_interval(n, cur_year, cur_month) if not done else None
        if x:
            start, end, rule = x
            rec.update({
                'time_kind': 'intervalle', 'precision': 'jour',
                'annee': start.year, 'mois': start.month, 'jour': start.day,
                'datetime_auto': fmt_dt(start, 'date'),
                'time_code_auto': make_code(start.year, start.month, start.day),
                'jour_romanesque': day_rom(start),
                'debut_auto': fmt_dt(start, 'date'),
                'fin_auto': fmt_dt(end, 'date'),
                'regle': rule,
            })
            # Un intervalle daté peut devenir un nouveau contexte narratif.
            if should_update_context(start.year, narrative_year):
                narrative_year = narrative_year or start.year
                cur_year, cur_month = start.year, start.month
                cur_date = pd.Timestamp(start.date())
                cur_dt = datetime(start.year, start.month, start.day)
            done = True

        # 3. Dates absolues : 1872, 2 octobre, samedi 21 décembre.
        if not done:
            x = absolute_date(n, cur_year, cur_month)
            if isinstance(x, tuple):
                val, rule = x
                if isinstance(val, datetime):
                    cl = parse_clock(n)
                    if cl and not looks_like_duration(n):
                        h, mi = cl
                        val = val.replace(hour=h, minute=mi)
                        prec = 'datetime'
                        kind = 'date_heure'
                    else:
                        h = mi = np.nan
                        prec = 'jour'
                        kind = 'date_absolue'
                    rec.update({
                        'time_kind': kind, 'precision': prec,
                        'annee': val.year, 'mois': val.month, 'jour': val.day,
                        'heure': h, 'minute': mi,
                        'datetime_auto': fmt_dt(val, prec),
                        'time_code_auto': make_code(val.year, val.month, val.day, 0 if pd.isna(h) else h, 0 if pd.isna(mi) else mi),
                        'jour_romanesque': day_rom(val),
                        'regle': rule,
                    })
                    # Une date explicite met à jour l'ancre si elle appartient au fil narratif.
                    if should_update_context(val.year, narrative_year):
                        if narrative_year is None:
                            narrative_year = val.year
                        cur_year, cur_month = val.year, val.month
                        cur_date = pd.Timestamp(val.date())
                        cur_dt = val if prec == 'datetime' else datetime(val.year, val.month, val.day)
                    done = True
                elif isinstance(val, dict):
                    y = val['year']
                    mo = val.get('month', np.nan)
                    prec = val.get('precision', 'annee')
                    rec.update({
                        'time_kind': 'date_absolue', 'precision': prec,
                        'annee': y, 'mois': mo,
                        'datetime_auto': f"{y:04d}-{int(mo):02d}" if pd.notna(mo) else str(y),
                        'time_code_auto': make_code(y, 0 if pd.isna(mo) else mo),
                        'regle': rule,
                    })
                    # Pour une année ou un mois seul, on met à jour seulement ce que l'on sait.
                    if should_update_context(y, narrative_year):
                        if narrative_year is None:
                            narrative_year = y
                        cur_year = y
                        if pd.notna(mo):
                            cur_month = int(mo)
                    done = True

        # 4. Fréquences : chaque jour, tous les mois, à des heures fixes.
        if not done and (any(x in n for x in ['chaque jour', 'tous les jours', 'jour par jour', 'tous les mois', 'chaque mois']) or re.search(r"\ba des heures\b|\ba de certaines heures\b", n)):
            rec.update({'time_kind': 'frequence', 'precision': 'non_instancie', 'regle': 'frequence'})
            done = True

        # 5. Durées non ponctuelles : depuis deux jours, pendant vingt-quatre heures.
        if not done and looks_like_duration(n) and not is_relative_offset_duration(n):
            # Les durées ancrées utilisent la date courante comme point de calcul.
            base = parse_anchor_dt(cur_dt, cur_date)
            xdur = anchored_duration_interval(n, base)
            if xdur:
                info = xdur['info']
                start = xdur['start']
                end = xdur['end']
                rec.update({
                    'time_kind': 'duree',
                    'precision': 'intervalle' if start is not None and end is not None else 'quantite',
                    'duration_value': info.get('duration_value', np.nan),
                    'duration_unit': info.get('duration_unit', np.nan),
                    'duration_relation': info.get('duration_relation', np.nan),
                    'regle': xdur['rule'],
                })
                if start is not None and end is not None:
                    rec.update({
                        'annee': end.year, 'mois': end.month, 'jour': end.day,
                        'heure': end.hour, 'minute': end.minute,
                        'datetime_auto': fmt_dt(end, 'datetime'),
                        'time_code_auto': make_code(end.year, end.month, end.day, end.hour, end.minute),
                        'jour_romanesque': day_rom(end),
                        'debut_auto': fmt_dt(start, 'datetime'),
                        'fin_auto': fmt_dt(end, 'datetime'),
                    })
                done = True

        # 6. Relatifs : le lendemain, la veille, trois heures après.
        if not done:
            # Certaines mentions comme 'ce jour-là' peuvent être précisées plus loin dans la phrase.
            lookahead_dt = sentence_next_dates.get((int(r.paragraph_ID), int(r.sentence_ID)))
            x = relative(n, cur_dt, cur_date, cur_year, cur_month, lookahead_dt=lookahead_dt)
            if isinstance(x, tuple):
                val, rule = x
                if isinstance(val, dict):
                    y = val['year']
                    mo = val.get('month', np.nan)
                    prec = val.get('precision', 'annee')
                    rec.update({
                        'time_kind': 'relatif', 'precision': prec,
                        'annee': y, 'mois': mo,
                        'datetime_auto': f"{y:04d}-{int(mo):02d}" if pd.notna(mo) else str(y),
                        'time_code_auto': make_code(y, 0 if pd.isna(mo) else mo),
                        'regle': rule,
                        })
                else:
                    dt = val
                    rec.update({
                        'time_kind': 'relatif', 'precision': 'datetime',
                        'annee': dt.year, 'mois': dt.month, 'jour': dt.day,
                        'heure': dt.hour, 'minute': dt.minute,
                        'datetime_auto': fmt_dt(dt, 'datetime'),
                        'time_code_auto': make_code(dt.year, dt.month, dt.day, dt.hour, dt.minute),
                        'jour_romanesque': day_rom(dt),
                        'regle': rule,
                        })
                    # Un relatif calculé peut devenir le nouveau contexte courant.
                    if should_update_context(dt.year, narrative_year):
                        cur_year, cur_month = dt.year, dt.month
                        cur_date = pd.Timestamp(dt.date())
                        # Une heure seule affine le contexte, mais ne change pas forcément le jour.
                    cur_dt = dt
                done = True

        # 7. Heures seules : à minuit, onze heures et demie.
        if not done and not looks_like_duration(n):
            cl = parse_clock(n)
            if cl:
                h, mi = cl
                rec.update({'time_kind': 'heure', 'precision': 'heure', 'heure': h, 'minute': mi, 'regle': 'heure_seule'})
                if cur_date is not None:
                    dt = datetime(cur_date.year, cur_date.month, cur_date.day, h, mi)
                    rec.update({
                        'annee': dt.year, 'mois': dt.month, 'jour': dt.day,
                        'datetime_auto': fmt_dt(dt, 'datetime'),
                        'time_code_auto': make_code(dt.year, dt.month, dt.day, h, mi),
                        'jour_romanesque': day_rom(dt),
                    })
                    cur_dt = dt
                elif cur_year:
                    rec.update({'annee': cur_year, 'time_code_auto': make_code(cur_year, 0, 0, h, mi)})
                done = True

        # 8. Durée simple non ancrée : 7 jours, quelques heures.
        if not done and looks_like_duration(n):
            info = parse_duration_info(n) or {}
            rec.update({
                'time_kind': 'duree', 'precision': 'quantite', 'regle': 'duree_quantite',
                'duration_value': info.get('duration_value', np.nan),
                'duration_unit': info.get('duration_unit', np.nan),
                'duration_relation': info.get('duration_relation', np.nan),
            })
            done = True

        # 9. Dernier classement : vague, faux positif ou non parsé.
        if not done:
            kind, precision = classify_leftover(n)
            rec.update({'time_kind': kind, 'precision': precision, 'regle': kind})

        rows.append(rec)

    parsed = pd.DataFrame(rows)
    return parsed


## 6) Lancer la normalisation

Cette cellule applique la fonction principale à toutes les entités `TIME` du fichier automatique.

Le CSV principal garde seulement les colonnes nécessaires pour la suite : position de l'entité, texte original, type de temps, date calculée, intervalle éventuel, durée éventuelle et règle utilisée.

Les colonnes internes de contrôle restent dans `parsed_full`, mais ne sont pas exportées dans le fichier principal. Cela permet de garder un CSV final lisible tout en conservant assez d'informations dans le notebook pour diagnostiquer les erreurs.


In [18]:
# On lit les entités automatiques et on garde uniquement les TIME.
# Le tri par start_token respecte l'ordre du récit.
ent = pd.read_csv(ENTITIES, sep='\t')
time = ent[ent['cat'] == 'TIME'].copy().sort_values('start_token').reset_index(drop=True)

# Table complète, avec quelques colonnes internes utiles au contrôle.
# parsed_full contient plus d'informations que le CSV final.
parsed_full = parse_time_table(time)

# Colonnes gardées dans le CSV principal.
# On évite d'exporter les colonnes purement internes pour garder la suite lisible.
# Elles suffisent pour rattacher ensuite les TIME aux triptyques.
EXPORT_COLS = [
    'entity_id', 'paragraph_ID', 'sentence_ID', 'start_token', 'end_token', 'text',
    'time_kind', 'precision',
    'annee', 'mois', 'jour', 'heure', 'minute',
    'datetime_auto', 'time_code_auto', 'jour_romanesque',
    'debut_auto', 'fin_auto',
    'duration_value', 'duration_unit', 'duration_relation',
    'regle',
]

parsed = parsed_full[EXPORT_COLS].copy()
# Ce fichier est celui que le notebook de rattachement aux triptyques va utiliser.
parsed.to_csv(OUT, index=False, encoding='utf-8')

# Fichier court pour inspecter des exemples de chaque catégorie.
examples = []
for kind in parsed['time_kind'].dropna().unique():
    sub = parsed[parsed['time_kind'] == kind].head(20).copy()
    sub.insert(0, 'groupe', kind)
    examples.append(sub[['groupe'] + EXPORT_COLS])

examples = pd.concat(examples, ignore_index=True)
examples.to_csv(OUT_EXEMPLES, index=False, encoding='utf-8')

print('Fichier écrit :', OUT)
print('Mentions TIME :', len(parsed))
print('Mentions avec code temporel :', parsed['time_code_auto'].notna().sum())
print(parsed['time_kind'].value_counts(dropna=False))


Fichier écrit : ../results/csv_triptyques/time_mentions_auto.csv
Mentions TIME : 868
Mentions avec code temporel : 649
time_kind
relatif                289
heure                  220
duree                  181
date_absolue            92
moment_vague            32
non_parse               17
reference_evenement     14
frequence                7
intervalle               6
relatif_non_calcule      5
garbage_probable         5
Name: count, dtype: int64


## 7) Contrôler les cas difficiles

Cette cellule est une cellule de lecture, pas une cellule de transformation.

Elle affiche les mentions qui restent non parsées, trop vagues, relatives mais non calculables, ou probablement mal annotées comme `TIME`. Elle sert à repérer les règles à améliorer sans modifier directement les données produites.


In [19]:
# Contrôle des cas difficiles.
# Cette table aide à repérer les mentions pour lesquelles une règle manque encore.
# parsed_full garde quelques colonnes internes non exportées dans le CSV final.
controle_cols = [
    'start_token', 'text', 'text_norm', 'time_kind', 'precision',
    'datetime_auto', 'time_code_auto', 'debut_auto', 'fin_auto',
    'duration_value', 'duration_unit', 'anchor_before', 'regle'
]

# On affiche seulement les catégories qui restent difficiles à normaliser.
parsed_full[
    parsed_full['time_kind'].isin([
        'non_parse',
        'relatif_non_calcule',
        'reference_evenement',
        'garbage_probable',
    ])
][controle_cols].head(80)


,start_token,text,text_norm,time_kind,precision,datetime_auto,time_code_auto,debut_auto,fin_auto,duration_value,duration_unit,anchor_before,regle
2,762,depuis de longues années,depuis de longues annees,relatif_non_calcule,non_instancie,NaN,NaN,NaN,NaN,NaN,NaN,1872,relatif_non_calcule
91,8398,qui,qui,garbage_probable,non_temporel,NaN,NaN,NaN,NaN,NaN,NaN,1872-10-07 03:00,garbage_probable
142,12633,le jour même de notre départ,le jour meme de notre depart,reference_evenement,non_instancie,NaN,NaN,NaN,NaN,NaN,NaN,1872-10-02 03:00,reference_evenement
157,15038,seize cent cinquante,seize cent cinquante,non_parse,non_instancie,NaN,NaN,NaN,NaN,NaN,NaN,1872-10-14 00:00,non_parse
164,15227,deux mille ans,deux mille ans,non_parse,non_instancie,NaN,NaN,NaN,NaN,NaN,NaN,1872-10-14 15:00,non_parse
176,15770,laquelle,laquelle,garbage_probable,non_temporel,NaN,NaN,NaN,NaN,NaN,NaN,1872-10-20 00:00,garbage_probable
191,17779,la dernière révolte des cipayes,la derniere revolte des cipayes,reference_evenement,non_instancie,NaN,NaN,NaN,NaN,NaN,NaN,1872-10-20 22:00,reference_evenement
192,17793,depuis son jeune âge,depuis son jeune age,relatif_non_calcule,non_instancie,NaN,NaN,NaN,NaN,NaN,NaN,1872-10-20 22:00,relatif_non_calcule
201,19041,jusqu' à son arrivée à bombay,jusqu' a son arrivee a bombay,reference_evenement,non_instancie,NaN,NaN,NaN,NaN,NaN,NaN,1872-10-21 12:00,reference_evenement
305,31679,trois mille cinq cents milles,trois mille cinq cents milles,non_parse,non_instancie,NaN,NaN,NaN,NaN,NaN,NaN,1872-10-20 01:00,non_parse


## 8) Vérifier les durées

Cette cellule isole les durées et les intervalles pour vérifier qu'ils sont bien traités.

Il faut surtout contrôler deux choses : les durées simples, qui doivent rester des quantités, et les durées ancrées, qui doivent produire un début et une fin.


In [20]:
# Résumé rapide des durées et des intervalles.
# On vérifie surtout que les durées ancrées ont bien debut_auto et fin_auto.
parsed[
    parsed['time_kind'].isin(['duree', 'intervalle'])
][[
    'entity_id', 'text', 'time_kind', 'precision',
    'time_code_auto', 'debut_auto', 'fin_auto',
    'duration_value', 'duration_unit', 'duration_relation',
    'regle'
]].head(40)


,entity_id,text,time_kind,precision,time_code_auto,debut_auto,fin_auto,duration_value,duration_unit,duration_relation,regle
6,6,sur vingt-quatre heures,duree,quantite,NaN,NaN,NaN,24.0,heure,sur,duree_non_ancree
9,9,entre onze heures et onze heures et demie,intervalle,heure,1872-10-02-11-00,1872-10-02 11:00,1872-10-02 11:30,NaN,NaN,NaN,intervalle_heure
13,13,voilà cinq ans,duree,intervalle,1872-10-02-11-30,1867-10-02 11:30,1872-10-02 11:30,5.0,annee,depuis,duree_depuis_intervalle
18,18,pendant les quelques instants,duree,quantite,NaN,NaN,NaN,NaN,NaN,NaN,duree_quantite
19,19,depuis cinq ans,duree,intervalle,1872-10-02-00-00,1867-10-02 00:00,1872-10-02 00:00,5.0,annee,depuis,duree_depuis_intervalle
38,38,un jour,duree,quantite,NaN,NaN,NaN,1.0,jour,duree_simple,duree_quantite
44,44,trois mois,duree,quantite,NaN,NaN,NaN,3.0,mois,duree_simple,duree_quantite
45,45,en quatre-vingts jours seulement,duree,intervalle,1872-12-18-00-00,1872-09-29 00:00,1872-12-18 00:00,80.0,jour,en,duree_forward_intervalle_en
46,46,7 jours,duree,quantite,NaN,NaN,NaN,7.0,jour,duree_simple,duree_quantite
47,47,13 jours,duree,quantite,NaN,NaN,NaN,13.0,jour,duree_simple,duree_quantite
